# ▶️ Rodar o projeto no Google Colab (demo end-to-end)

**Tech Challenge Fase 3 — Assistente Médico Inteligente (Flamers Team)**

Este notebook **executa** o projeto (RAG + LLM fine-tunada + agentes + UI Gradio).
Ele **não treina** nada — o fine-tuning está em `notebooks/02_finetuning.ipynb`.

### Pré-requisitos
1. **Ambiente de execução → Alterar o tipo → GPU** (T4 / L4 / A100 servem).
2. Pasta no Google Drive: `MyDrive/techchallenge_fase3/data/` com `raw/` (e `processed/` se tiver o `chroma_index`).
3. Adapter LoRA publicado no HuggingFace: `michelleAnogueira/biomistral-medquad-lora`.
4. As correções deste commit já **pushadas** no repositório (o notebook faz `git clone` / `git pull`).

Rode as células **em ordem, de cima para baixo**. Não precisa reiniciar o kernel.

In [ ]:
# ======================================================================
# CONFIGURAÇÃO
# ======================================================================
import os

REPO_URL     = "https://github.com/Flamers-Team/MedAssistPro.git"  # troque pelo seu fork se ainda não deu push
PROJECT_DIR  = "/content/MedAssistPro"
DRIVE_DIR    = "/content/drive/MyDrive/techchallenge_fase3"

LLM_MODEL    = "michelleAnogueira/biomistral-medquad-lora"  # repo do adapter LoRA no HuggingFace
LLM_MOCK     = False   # True = respostas sintéticas (testar a pipeline/UI sem GPU)
GRADIO_SHARE = True    # gera link público *.gradio.live (bom para o vídeo)
UI_TRANSLATE = True    # carrega o tradutor PT-BR <-> EN (MarianMT) na UI

INDEX_REBUILD_LIMIT = 20000  # nº de bulas a indexar SE precisar reconstruir o índice RAG

os.environ["LLM_MODEL"]    = LLM_MODEL
os.environ["LLM_MOCK"]     = "1" if LLM_MOCK else "0"
os.environ["GRADIO_SHARE"] = "1" if GRADIO_SHARE else "0"
os.environ["UI_TRANSLATE"] = "1" if UI_TRANSLATE else "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Config OK.")

In [ ]:
# ======================================================================
# 1/8 — Checar GPU
# ======================================================================
import subprocess, torch

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip() or "⚠️ SEM GPU detectada")
print("torch:", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())

assert torch.cuda.is_available() or os.environ["LLM_MOCK"] == "1", (
    "Ative a GPU (Ambiente de execução → Alterar o tipo) ou defina LLM_MOCK = True na célula de config."
)

In [ ]:
# ======================================================================
# 2/8 — Montar o Google Drive
# ======================================================================
from google.colab import drive
drive.mount("/content/drive")

assert os.path.isdir(DRIVE_DIR), f"Não achei {DRIVE_DIR} no seu Drive."
data_dir = os.path.join(DRIVE_DIR, "data")
assert os.path.isdir(data_dir), f"Não achei {data_dir} no seu Drive."
print("Drive OK.")
print("data/ ->", os.listdir(data_dir))
print("data/raw/ ->", os.listdir(os.path.join(data_dir, "raw")))

In [ ]:
# ======================================================================
# 3/8 — Clonar / atualizar o repositório
# ======================================================================
import sys

if not os.path.isdir(os.path.join(PROJECT_DIR, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"], check=False)

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("CWD:", os.getcwd())
print(sorted(os.listdir(".")))
assert os.path.exists("requirements-colab.txt"), (
    "O repositório clonado não tem requirements-colab.txt — faça commit/push das correções "
    "ou aponte REPO_URL para o seu fork."
)

In [ ]:
# ======================================================================
# 4/8 — Instalar dependências (numpy 2.x nativo do Colab, SEM downgrade)
# ======================================================================
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("\n✅ Dependências instaladas.")
print("   Se o pip pedir para REINICIAR a sessão: reinicie e rode a partir da célula 3 (clone é idempotente).")

In [ ]:
# ======================================================================
# 5/8 — (OPCIONAL) Login no HuggingFace
# Só é necessário se 'michelleAnogueira/biomistral-medquad-lora' for PRIVADO.
# Guarde o token em Secrets do Colab (🔑 no menu esquerdo) com o nome HF_TOKEN.
# ======================================================================
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        from huggingface_hub import login
        login(_tok)
        print("HuggingFace login OK.")
    else:
        print("Sem HF_TOKEN nos Secrets — ok se o repo do adapter for público.")
except Exception as e:
    print("Pulei o login HF:", e)

In [ ]:
# ======================================================================
# 6/8 — Copiar data/ do Drive para o projeto
# ======================================================================
import shutil

DRIVE_DATA = os.path.join(DRIVE_DIR, "data")
DEST_DATA  = os.path.join(PROJECT_DIR, "data")

for sub in ("raw", "processed"):
    s = os.path.join(DRIVE_DATA, sub)
    if os.path.isdir(s):
        shutil.copytree(s, os.path.join(DEST_DATA, sub), dirs_exist_ok=True)
        print(f"copiado: data/{sub}/")

_raw = os.path.join(DEST_DATA, "raw")
_proc = os.path.join(DEST_DATA, "processed")
print("raw/      ->", os.listdir(_raw) if os.path.isdir(_raw) else "—")
print("processed/ ->", os.listdir(_proc) if os.path.isdir(_proc) else "—")

In [ ]:
# ======================================================================
# 7/8 — Garantir o índice RAG (ChromaDB)
#  - Usa o chroma_index que veio do Drive.
#  - Só reconstrói (rápido, na GPU) se ele não existir OU não abrir nesta
#    versão do chromadb. NENHUMA reindexação de 20 min como antes.
# ======================================================================
CHROMA = os.path.join(PROJECT_DIR, "data", "processed", "chroma_index")

def _index_ok():
    if not os.path.exists(os.path.join(CHROMA, "chroma.sqlite3")):
        return False
    try:
        from src.rag.retriever import Retriever
        r = Retriever()
        c = r.collections.get("chatbulario")
        n = c.count() if c else 0
        print(f"Índice do Drive OK — coleção 'chatbulario': {n:,} docs")
        return n > 0
    except Exception as e:
        print("Índice do Drive não abriu nesta versão do chromadb:", e)
        return False

if _index_ok():
    print("✅ Sem reindexação necessária.")
else:
    print(f"\n🔧 Reconstruindo índice ({INDEX_REBUILD_LIMIT} bulas, embeddings na GPU ~1 min)...")
    shutil.rmtree(CHROMA, ignore_errors=True)
    subprocess.run(
        [sys.executable, "src/rag/build_index_chatbulario.py", str(INDEX_REBUILD_LIMIT)],
        check=True,
    )
    try:
        dst = os.path.join(DRIVE_DIR, "chroma_index_v2")
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(CHROMA, dst)
        print("Índice novo salvo no Drive:", dst,
              "(copie para data/processed/chroma_index/ no Drive p/ reusar)")
    except Exception as e:
        print("(não consegui cachear o índice no Drive:", e, ")")

In [ ]:
# ======================================================================
# 8/8 — Teste rápido: RAG + LLM respondendo
# ======================================================================
os.chdir(PROJECT_DIR)

from src.rag.retriever import Retriever
r = Retriever()
hits = r.retrieve_chatbulario("efeitos colaterais de paracetamol", k=2)
print("RAG OK —", len(hits), "resultado(s).")
if hits:
    print("   ex.:", hits[0]["content"][:140].replace("\n", " "), "...")

from src.llm.client import get_llm
llm = get_llm()
print("\nLLM (mock =", llm.use_mock, ") respondendo...\n")
print(llm.invoke([
    {"role": "system", "content": "You are a medical assistant."},
    {"role": "user", "content": "What are the symptoms of type 2 diabetes?"},
])[:400])

In [ ]:
# ======================================================================
# SUBIR A INTERFACE GRADIO
#  Aguarde ~1 min. Vai aparecer um link *.gradio.live.
#  Login: medico / demo123
#  Deixe ESTA célula rodando durante a demonstração.
#  A tradução PT-BR <-> EN aparece como um checkbox na aba "Consulta".
# ======================================================================
os.chdir(PROJECT_DIR)
os.environ["GRADIO_SHARE"] = "1" if GRADIO_SHARE else "0"
os.environ["LLM_MODEL"]    = LLM_MODEL
os.environ["LLM_MOCK"]     = "1" if LLM_MOCK else "0"
os.environ["UI_TRANSLATE"] = "1" if UI_TRANSLATE else "0"

!python src/ui/gradio_app.py

## 🩹 Problemas comuns

| Sintoma | Solução |
|---|---|
| `pip` pediu **RESTART** | Ambiente de execução → Reiniciar sessão, e rode a partir da **célula 3**. |
| **OOM** ao carregar a LLM | Use GPU maior (L4 / A100) ou `LLM_MOCK = True` para validar o fluxo. |
| Adapter **privado** no HF | Coloque `HF_TOKEN` nos Secrets do Colab (célula 5). |
| 1ª carga da LLM demora | O base `BioMistral/BioMistral-7B` (~14 GB) é baixado 1x por sessão. |
| Índice do Drive não abre | A célula 7 reconstrói sozinha (~1 min na GPU). |
| UI sem link público | Confirme `GRADIO_SHARE = True` na config. |
| `git pull` falhou (working tree suja) | `!cd /content/MedAssistPro && git reset --hard origin/main` e rode de novo. |